<a href="https://colab.research.google.com/github/nahmeddn-sys/GENAI/blob/main/Assignment_3_Perform_training_of_an_LLM_built_from_scratch_using_transformation_library_which_is_of_reasonable_size_using_a_suitable_dataset_such_as_Wikipedia_text_and_test_the_LLM_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Introduction**

Large Language Models (LLMs) are advanced deep learning models designed to understand and generate human-like text. These models are typically built using the Transformer architecture, which allows them to learn relationships between words in a sentence through a mechanism called self-attention. By training on large text datasets such as Wikipedia or similar corpora, an LLM learns patterns of language, grammar, and context. In this implementation, a small transformer-based language model inspired by the GPT architecture is trained from scratch using the Hugging Face Transformers library on a subset of a Wikipedia-derived dataset (Wikitext). Since training a full-scale LLM requires massive computational resources, a lightweight version of the model is trained using a free Google Colab GPU, demonstrating the fundamental principles of LLM training while remaining computationally feasible.

**Explanation**

The training pipeline begins by loading and preparing a text dataset, which is then processed using a tokenizer to convert natural language into numerical tokens that the model can understand. These tokenized sequences are fed into a transformer model configured with multiple attention layers that learn contextual relationships between words. During training, the model uses a causal language modeling objective, meaning it learns to predict the next word in a sequence based on the preceding words. The training process is managed using the Hugging Face Trainer API, which handles batching, optimization, and GPU acceleration. After training, the model is evaluated by providing prompts and generating new text, demonstrating how it has learned linguistic patterns from the dataset. Although the model is smaller and trained on limited data, it successfully illustrates the core workflow involved in building, training, and testing transformer-based language models.

**Install Libraries**

In [ ]:
!pip install transformers datasets accelerate -q

**Import Libraries**

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    GPT2Config,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

**Load Dataset (Wikipedia-style dataset)**

Wikitext-2 is used because it is derived from Wikipedia and trains faster.

In [ ]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

train_dataset = dataset["train"]

# remove very short lines
train_dataset = train_dataset.filter(lambda x: len(x["text"]) > 50)

print(train_dataset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36718 [00:00<?, ? examples/s]

{'text': ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n'}


**Load Tokenizer**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# GPT2 has no pad token
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

**Tokenization Function**

In [ ]:
def tokenize_function(example):

    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

**Tokenize Dataset**

In [ ]:
tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","labels"]
)

Map:   0%|          | 0/16323 [00:00<?, ? examples/s]

**Build Transformer Model**

In [ ]:
config = GPT2Config(

    vocab_size=len(tokenizer),
    n_positions=256,

    n_embd=384,      # embedding dimension
    n_layer=6,       # transformer layers
    n_head=6,        # attention heads

    pad_token_id=tokenizer.eos_token_id
)

model = GPT2LMHeadModel(config)

print("Model parameters:", model.num_parameters())

Model parameters: 30044544


**Data Collator**

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

**Training Configuration**

In [ ]:
training_args = TrainingArguments(

    output_dir="mini_llm_output",

    num_train_epochs=3,

    per_device_train_batch_size=4,

    gradient_accumulation_steps=2,

    logging_steps=100,

    save_steps=500,

    fp16=torch.cuda.is_available()
)

**Trainer**

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_dataset,

    data_collator=data_collator
)

**Train the Model**

In [ ]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,9.375267
200,7.824097
300,7.230275
400,7.069805
500,6.993564
600,6.932122
700,6.891031
800,6.857737
900,6.785891
1000,6.737556


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6123, training_loss=6.378067051490187, metrics={'train_runtime': 769.8162, 'train_samples_per_second': 63.611, 'train_steps_per_second': 7.954, 'total_flos': 800870359891968.0, 'train_loss': 6.378067051490187, 'epoch': 3.0})

**Save Model**

In [ ]:
trainer.save_model("mini_wiki_llm")
tokenizer.save_pretrained("mini_wiki_llm")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('mini_wiki_llm/tokenizer_config.json', 'mini_wiki_llm/tokenizer.json')

Verify the path or folder


In [ ]:
!ls mini_wiki_llm

config.json		model.safetensors      tokenizer.json
generation_config.json	tokenizer_config.json  training_args.bin


**Test the LLM**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

input_text = "Anarchism"

inputs = tokenizer(input_text, return_tensors="pt")

# Move inputs to GPU
inputs = {key: value.to(device) for key, value in inputs.items()}

output = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Anarchism and 's name was a original album in the end of the song at the album . On March 2009 , and the video was named an first on the song . She was a new game to the title of the album , and had the episode of the song . The song , it was the "
